# 07 — Linked Lists

## Why This Matters for DSA
This notebook closes the loop on two techniques you've already learned in a different setting: fast/slow pointers (`01_Arrays_and_Two_Pointers`) return to their most natural home, and the recursion patterns from `02_Recursion` get applied to a genuinely pointer-based structure for the first time. Linked lists are also where "no random access" stops being an abstract cost and starts being something you have to actively design around — every technique here (the dummy head node especially) exists specifically to work around that constraint cleanly.

## Prerequisites
- `01_Arrays_and_Two_Pointers` (Phase 02) — fast/slow pointers, originally introduced via an array-as-implicit-linked-structure trick; this notebook applies the same technique to real linked lists
- `02_Recursion` (Phase 01) — the recursive reversal in this notebook is a direct, concrete application of "trust the recursion" reasoning

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain why linked lists trade O(1) random access (a vector's strength) for O(1) insertion/deletion at a known position (a vector's weakness)
- Implement insertion and deletion at the head, tail, and an arbitrary position
- Reverse a linked list both iteratively (O(1) space) and recursively (O(n) space), and explain the space difference concretely
- Apply fast/slow pointers to find a list's middle and detect a cycle
- Use the dummy head node technique to eliminate special-case code for operations that might affect the original head
- Merge two sorted linked lists by relinking existing nodes, without allocating a new result structure
- Implement a doubly linked list and explain why it enables O(1) removal given a node pointer, which a singly linked list cannot do
- Judge when a linked list is the right structure versus when an array-based structure fits better

## Checklist Before Moving On
- [ ] I can state the core trade-off between arrays and linked lists in one sentence
- [ ] I can reverse a linked list iteratively without looking up the three-pointer pattern
- [ ] I can explain, concretely, why recursive reversal costs O(n) space while iterative costs O(1)
- [ ] I can explain why fast/slow pointers finding the middle works, using the same 2x-speed reasoning as the array version
- [ ] I can explain what problem the dummy head node solves, not just how to use it
- [ ] I can explain why doubly (not singly) linked lists are the structure behind LRU caches

## Table of Contents
1. Singly Linked List Fundamentals — Nodes, Traversal, and the Core Trade-off
2. Insertion and Deletion — Head, Tail, and by Value
3. Reversing a Linked List — Iterative and Recursive
4. Fast and Slow Pointers Revisited — Middle and Cycle Detection
5. The Dummy Head Node Technique
6. Merging Two Sorted Linked Lists
7. Doubly Linked List — O(1) Removal With a Known Node
8. When a Linked List Is (and Isn't) the Right Tool
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. Singly Linked List Fundamentals — Nodes, Traversal, and the Core Trade-off

### The Why
Every technique in this notebook builds on one structural fact: a linked list has no contiguous memory block, and no operator[]. Internalizing WHY that one fact drives every design choice here — the dummy head node, the fast/slow pointer trick, everything — is more valuable than memorizing any single operation.

### Core Mechanism
- A node holds a value and a POINTER to the next node. Nodes can live anywhere in memory — what makes it a "list" is purely the chain of pointers connecting them, not physical adjacency.
- **Traversal is always O(n)** to reach an arbitrary position — the only way to reach node `i` is to walk through nodes `0` to `i-1` first. There is no equivalent of `vector`'s O(1) `operator[]`.
- **The core trade-off against `vector`:** a vector gives O(1) random access but O(n) insertion/deletion in the middle (everything after the insertion point must shift, `01_Arrays_and_Two_Pointers`). A linked list flips this exactly: O(n) to reach an arbitrary position, but O(1) insertion/deletion ONCE you're already holding a pointer to the right spot — no shifting required, since inserting or removing a node is just rewiring a couple of pointers.
- Neither structure is strictly better — they suit different access patterns. A workload dominated by random lookups favors `vector`; a workload dominated by insertions/deletions at already-known positions (like the middle of a splice-heavy sequence) favors a linked list.

### Common Pitfalls
- Choosing a linked list "because insertion is O(1)" without noticing that REACHING the insertion point in the first place is O(n) unless you already hold a pointer to it — the O(1) insertion advantage only pays off when you're not paying an O(n) search to get there first.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// A linked list node holds a value and a POINTER to the next node -- unlike a vector,
// there is NO contiguous memory block. Each node can live ANYWHERE in memory; what
// makes it a "list" is purely the chain of pointers connecting them.
struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

// build a list from a vector, for convenience in testing
ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);          // see Section 6 for why a dummy node is a recurring trick
    ListNode* tail = &dummy;
    for (int v : vals) {
        tail->next = new ListNode(v);
        tail = tail->next;
    }
    return dummy.next;
}

void printList(ListNode* head) {
    while (head != nullptr) {
        cout << head->val;
        if (head->next != nullptr) cout << " -> ";
        head = head->next;
    }
    cout << "\n";
}

int main() {
    vector<int> vals{1, 2, 3, 4, 5};
    ListNode* head = buildList(vals);
    printList(head);

    // TRAVERSAL: the only way to reach node i is to walk through nodes 0..i-1 first --
    // there's no operator[] equivalent, no O(1) random access, unlike vector. Traversal
    // is always O(n) to reach an arbitrary position.
    int target = 3;
    ListNode* cur = head;
    int steps = 0;
    while (cur != nullptr && cur->val != target) {
        cur = cur->next;
        steps++;
    }
    cout << "found " << target << " after " << steps << " traversal steps\n";

    // WHY THIS TRADE-OFF CAN STILL BE WORTH IT: insertion/deletion at a KNOWN position
    // (given a pointer to it) is O(1) -- no shifting of subsequent elements required,
    // unlike a vector's O(n) insert/erase in the middle (01_Arrays_and_Two_Pointers).
    // The trade is: linked lists give up O(1) random access (vector's strength) in
    // exchange for O(1) insertion/deletion at a known point (vector's weakness) --
    // neither structure is strictly better, they're suited to different access patterns.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Insertion and Deletion — Head, Tail, and by Value

### The Why
These three operations look similar but have genuinely different costs, and the difference is a direct consequence of Section 1's core trade-off — worth seeing concretely rather than taking on faith.

### Core Mechanism
- **Insert at head: O(1).** Point the new node's `next` at the current head, then update whatever tracks "head" to the new node. No traversal needed — you already have (or are given) a pointer to the head by definition.
- **Insert at tail: O(n) without a maintained tail pointer.** You must traverse the entire list to find the last node before you can attach to it. Many real implementations (including `std::list`) maintain a separate tail pointer specifically to make this O(1) too — worth knowing that's a deliberate design choice, not a given.
- **Delete by value: O(n) to FIND it, O(1) to actually remove it once found.** The search is the expensive part (must traverse to locate the matching value); the removal itself is a single pointer rewire (`prev->next = cur->next`), skipping over the deleted node entirely.
- All three deletion/insertion operations share a theme: once you're AT the right node (whether found by traversal or already held via a pointer), the actual structural change is O(1) — a small, fixed number of pointer updates, regardless of list length.

### Common Pitfalls
- Losing the reference to the rest of the list by overwriting a `next` pointer before saving what it currently points to — this is the single most common linked-list bug, and it recurs throughout this entire notebook (see Section 3's reversal in particular).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);
    ListNode* tail = &dummy;
    for (int v : vals) { tail->next = new ListNode(v); tail = tail->next; }
    return dummy.next;
}

void printList(ListNode* head) {
    while (head) { cout << head->val; if (head->next) cout << " -> "; head = head->next; }
    cout << "\n";
}

// INSERT AT HEAD: O(1) -- just repoint the new node's next to the old head, and update
// whatever variable is tracking "head" to point to the new node.
ListNode* insertAtHead(ListNode* head, int val) {
    ListNode* newNode = new ListNode(val);
    newNode->next = head;
    return newNode;   // caller must update their head pointer to this return value
}

// INSERT AT TAIL: O(n) UNLESS you separately maintain a tail pointer -- without one,
// you must traverse the entire list to find the last node before you can attach to it.
ListNode* insertAtTail(ListNode* head, int val) {
    ListNode* newNode = new ListNode(val);
    if (head == nullptr) return newNode;

    ListNode* cur = head;
    while (cur->next != nullptr) cur = cur->next;   // walk to the last node -- O(n)
    cur->next = newNode;
    return head;
}

// DELETE A NODE BY VALUE: O(n) to FIND it (must traverse), O(1) to actually remove it
// once found -- the removal itself is just repointing one pointer, skipping over the
// node being deleted.
ListNode* deleteByValue(ListNode* head, int val) {
    ListNode dummy(0);
    dummy.next = head;
    ListNode* prev = &dummy;
    ListNode* cur = head;

    while (cur != nullptr) {
        if (cur->val == val) {
            prev->next = cur->next;   // skip over `cur` -- this IS the deletion
            delete cur;
            break;
        }
        prev = cur;
        cur = cur->next;
    }
    return dummy.next;
}

int main() {
    vector<int> vals{2, 3, 4};
    ListNode* head = buildList(vals);

    head = insertAtHead(head, 1);
    printList(head);   // 1 -> 2 -> 3 -> 4

    head = insertAtTail(head, 5);
    printList(head);   // 1 -> 2 -> 3 -> 4 -> 5

    head = deleteByValue(head, 3);
    printList(head);   // 1 -> 2 -> 4 -> 5

    // WHY insert-at-head is O(1) but insert-at-tail is O(n) WITHOUT a maintained tail
    // pointer: the list only gives you direct access to whatever node you're currently
    // holding a pointer to. Reaching the tail from the head requires walking the full
    // chain; reaching the head requires no walk at all, since by definition you already
    // have (or are given) a pointer to it. Many real implementations maintain a separate
    // tail pointer specifically to make tail insertion O(1) too (see std::list, which
    // does exactly this internally)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Reversing a Linked List — Iterative and Recursive

### The Why
This is the single most common linked-list interview question, and it's the cleanest concrete illustration of `02_Recursion`'s abstract "iterative is O(1) space, recursive is O(n) space" point — both solutions here are O(n) TIME, but their SPACE costs genuinely differ, and you can see exactly why.

### Core Mechanism
- **Iterative (three-pointer technique):** at every step, track three things simultaneously — the node currently being reversed (`cur`), the node before it (`prev`, to point `cur` backward at), and the node after it (`next`, saved BEFORE rewiring `cur->next`, or the rest of the list becomes unreachable). When `cur` becomes null, `prev` is sitting on the new head (the old tail).
- **Recursive:** reverse everything AFTER the current node FIRST (trust the recursion completely — this is exactly the "trust the recursive call to handle the rest" instinct from `02_Recursion`), then fix up the current node's connections once that's done: the node right after `head` (now the tail of the already-reversed sub-list) gets pointed BACK at `head`, and `head` itself becomes the new tail, pointing to nothing.
- **The space difference, made concrete:** iterative uses O(1) extra space — three pointer variables, regardless of how long the list is. Recursive uses O(n) extra space — one stack frame per node, all alive simultaneously until the base case is hit and unwinding begins. This is `02_Recursion`'s abstract point about recursion's hidden stack cost, now demonstrated on a real, useful algorithm rather than a toy example.

### Common Pitfalls
- Forgetting to save `cur->next` into a temporary BEFORE overwriting `cur->next = prev` in the iterative version — without the save, the rest of the original list is lost the instant that line executes.
- In the recursive version, forgetting `head->next = nullptr` — without it, the original head (now the new tail) still points forward at what's now ITS OWN predecessor in the reversed list, creating a two-node cycle instead of a clean reversed list.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);
    ListNode* tail = &dummy;
    for (int v : vals) { tail->next = new ListNode(v); tail = tail->next; }
    return dummy.next;
}

void printList(ListNode* head) {
    while (head) { cout << head->val; if (head->next) cout << " -> "; head = head->next; }
    cout << "\n";
}

// ITERATIVE REVERSAL: the standard three-pointer technique. At every step, you need to
// remember THREE things at once: the node you're currently reversing, the node before
// it (to point the current node BACK at), and the node after it (or you'd lose the rest
// of the list the moment you rewire `cur->next`).
ListNode* reverseIterative(ListNode* head) {
    ListNode* prev = nullptr;
    ListNode* cur = head;

    while (cur != nullptr) {
        ListNode* next = cur->next;   // save this BEFORE overwriting cur->next, or the
                                        // rest of the list becomes unreachable
        cur->next = prev;             // reverse this node's pointer
        prev = cur;                   // advance prev
        cur = next;                   // advance cur, using the saved pointer
    }
    return prev;   // when cur becomes null, prev is sitting on the new head (old tail)
}

// RECURSIVE REVERSAL: reverse everything AFTER the current node first (trust the
// recursion), then fix up the current node's connection once that's done.
ListNode* reverseRecursive(ListNode* head) {
    if (head == nullptr || head->next == nullptr) return head;   // base case: 0 or 1 node

    ListNode* newHead = reverseRecursive(head->next);   // reverses everything after `head`,
                                                           // returns the NEW head of that
                                                           // reversed sub-list (the old tail)

    head->next->next = head;   // the node right after `head` (now the TAIL of the reversed
                                 // sub-list) should point BACK at `head`
    head->next = nullptr;      // `head` is now the new tail -- it must point to nothing

    return newHead;   // always propagate the same newHead up through every call
}

int main() {
    vector<int> vals1{1, 2, 3, 4, 5};
    ListNode* head1 = buildList(vals1);
    head1 = reverseIterative(head1);
    printList(head1);   // 5 -> 4 -> 3 -> 2 -> 1

    vector<int> vals2{1, 2, 3, 4, 5};
    ListNode* head2 = buildList(vals2);
    head2 = reverseRecursive(head2);
    printList(head2);   // 5 -> 4 -> 3 -> 2 -> 1

    // BOTH are O(n) time. Space differs: iterative is O(1) extra space (three pointers,
    // regardless of list length); recursive is O(n) extra space (the call stack), same
    // trade-off discussed generally back in 02_Recursion's space complexity section --
    // this is a clean, concrete example of that abstract point

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Fast and Slow Pointers Revisited — Middle and Cycle Detection

### The Why
`01_Arrays_and_Two_Pointers` introduced fast/slow pointers via a clever but somewhat indirect trick — treating array VALUES as implicit pointers. Here, the technique returns to its literal, most natural setting: a real linked list, where "next" is an actual pointer, not a computed index.

### Core Mechanism
- **Find the middle:** `slow` advances one step at a time, `fast` advances two. When `fast` reaches the end (or one before it), `slow` is exactly at the middle — for even-length lists, this specific implementation lands on the SECOND of the two middle nodes.
- **Detect a cycle (Floyd's algorithm, in its original setting):** same slow/fast movement; if `fast` ever becomes equal to `slow` again after both start moving, a cycle exists. If `fast` reaches a genuine end (`nullptr`) cleanly, there's no cycle.
- Both are **O(n) time, O(1) extra space** — identical complexity profile to the array version in `01_Arrays_and_Two_Pointers`, confirming this really is the same underlying technique, just applied to its more natural home. The reasoning ("fast covers twice the distance, so a cycle forces a meeting; no cycle means fast simply finishes first") is unchanged from the array version.

### Common Pitfalls
- Checking only `fast != nullptr` in the loop condition without also checking `fast->next != nullptr` — `fast->next->next` would then dereference a null pointer whenever the list has an even length and `fast` lands exactly on the last node, crashing instead of terminating cleanly.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);
    ListNode* tail = &dummy;
    for (int v : vals) { tail->next = new ListNode(v); tail = tail->next; }
    return dummy.next;
}

// FIND THE MIDDLE: slow moves 1 step, fast moves 2 steps -- when fast reaches the end,
// slow is exactly at the middle. This is the array-independent, general-purpose version
// of the fast/slow idea from 01_Arrays_and_Two_Pointers (which applied it to an ARRAY
// treated as an implicit linked structure) -- here it's a REAL linked list, the more
// natural home for this technique.
ListNode* findMiddle(ListNode* head) {
    ListNode* slow = head;
    ListNode* fast = head;

    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;
        fast = fast->next->next;
    }
    return slow;   // for even-length lists, this lands on the SECOND of the two middle nodes
}

// DETECT A CYCLE (Floyd's algorithm, applied to its original, most natural setting):
// if fast ever equals slow again after they've both started moving, there's a cycle.
bool hasCycle(ListNode* head) {
    ListNode* slow = head;
    ListNode* fast = head;

    while (fast != nullptr && fast->next != nullptr) {
        slow = slow->next;
        fast = fast->next->next;
        if (slow == fast) return true;   // they've met -- a cycle exists
    }
    return false;   // fast reached the end cleanly -- no cycle
}

int main() {
    vector<int> vals{1, 2, 3, 4, 5};
    ListNode* head = buildList(vals);
    cout << "middle of [1,2,3,4,5] = " << findMiddle(head)->val << "\n";   // expected 3

    vector<int> vals2{1, 2, 3, 4};
    ListNode* head2 = buildList(vals2);
    cout << "middle of [1,2,3,4] = " << findMiddle(head2)->val << "\n";    // expected 3 (2nd of 2 middles)

    cout << "hasCycle (no cycle) = " << hasCycle(head) << "\n";   // expected false

    // manually construct a cycle: last node points back to the 3rd node
    ListNode* a = new ListNode(1);
    ListNode* b = new ListNode(2);
    ListNode* c = new ListNode(3);
    a->next = b; b->next = c; c->next = b;   // c points back to b -- creates a cycle
    cout << "hasCycle (with cycle) = " << hasCycle(a) << "\n";   // expected true

    // both O(n) time, O(1) extra space -- exactly the same complexity profile as the
    // array version in 01_Arrays_and_Two_Pointers, confirming this is genuinely the
    // SAME technique, just applied to its more natural home

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. The Dummy Head Node Technique

### The Why
A huge fraction of linked-list bugs come from special-casing "what if the node I need to modify is the very first node" — because operations on the head often need to update whatever EXTERNAL variable is tracking "head," while operations elsewhere in the list only need to update a `next` pointer. The dummy head node eliminates this asymmetry entirely.

### Core Mechanism
- Create a fake node BEFORE the real head, pointing at it (`dummy.next = head`). Now "the node before the real head" always exists as an ordinary node — there's no longer a special case for "the operation needs to happen at the very front."
- **Removing elements matching a value, anywhere including a run at the front:** with a dummy node, the deletion logic (`prev->next = cur->next`) is IDENTICAL regardless of whether `cur` is the original head or somewhere in the middle — no `if (cur == head)` branch needed anywhere.
- **Removing the Nth node from the end:** combines the dummy node with a "maintain a gap" fast/slow variant — advance `fast` by `n` steps first to open a gap, then advance both together until `fast` falls off the end; `slow` ends up exactly one node before the target, ready for a clean removal, again with no special-casing for "what if the target is the original head."
- Return `dummy.next` (not the original `head` variable) at the end — this automatically reflects wherever the new head ended up, even in the case where the original head itself was removed.

### Common Pitfalls
- Declaring the dummy node on the heap (`new ListNode(0)`) and forgetting to free it, or more commonly, declaring it correctly as a local stack variable but then accidentally returning `&dummy` or a pointer into it instead of `dummy.next` — the dummy node itself is never part of the real list and should never leak into the returned result.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);
    ListNode* tail = &dummy;
    for (int v : vals) { tail->next = new ListNode(v); tail = tail->next; }
    return dummy.next;
}

void printList(ListNode* head) {
    while (head) { cout << head->val; if (head->next) cout << " -> "; head = head->next; }
    cout << "\n";
}

// THE DUMMY HEAD TECHNIQUE: create a fake node BEFORE the real head, so that "the node
// before the real head" always exists as a normal node -- eliminating special-case code
// for "what if we need to delete/modify the very first node."

// Remove all nodes with a given value, ANYWHERE in the list, including a run of matches
// at the very front.
ListNode* removeElements(ListNode* head, int val) {
    ListNode dummy(0);
    dummy.next = head;
    ListNode* prev = &dummy;   // prev ALWAYS has a valid "previous node" to work with,
                                 // even when we're about to delete the original head
    ListNode* cur = head;

    while (cur != nullptr) {
        if (cur->val == val) {
            prev->next = cur->next;   // same deletion logic regardless of WHERE cur is --
                                        // no special "is this the head?" branch needed
            ListNode* toDelete = cur;
            cur = cur->next;
            delete toDelete;
        } else {
            prev = cur;
            cur = cur->next;
        }
    }
    return dummy.next;   // dummy.next automatically reflects wherever the new head ended up,
                           // even if the original head itself got deleted
}

// Remove the Nth node from the END of the list -- another classic dummy-head use case,
// combined with the fast/slow technique's "maintain a gap" idea.
ListNode* removeNthFromEnd(ListNode* head, int n) {
    ListNode dummy(0);
    dummy.next = head;
    ListNode* fast = &dummy;
    ListNode* slow = &dummy;

    for (int i = 0; i < n; i++) fast = fast->next;   // open up a gap of n nodes

    while (fast->next != nullptr) {   // advance both until fast falls off the end --
        fast = fast->next;             // slow ends up exactly n nodes behind, i.e. right
        slow = slow->next;             // before the node that needs removing
    }

    ListNode* toDelete = slow->next;
    slow->next = slow->next->next;
    delete toDelete;

    return dummy.next;
}

int main() {
    vector<int> vals{1, 1, 2, 3, 1};
    ListNode* head = buildList(vals);
    head = removeElements(head, 1);
    printList(head);   // expected: 2 -> 3 (including the leading 1s removed cleanly)

    vector<int> vals2{1, 2, 3, 4, 5};
    ListNode* head2 = buildList(vals2);
    head2 = removeNthFromEnd(head2, 2);
    printList(head2);   // expected: 1 -> 2 -> 3 -> 5 (removed the 4, 2nd from the end)

    // WITHOUT a dummy node, both functions would need special-case code for "what if the
    // node(s) to remove include the original head" -- the dummy node makes that case
    // completely uniform with every other case, which is the entire value of the technique

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Merging Two Sorted Linked Lists

### The Why
This is the exact same two-pointer merge logic from `02_Sliding_Window_and_Prefix_Sum`'s array merge — but linked lists offer something arrays structurally can't: the merged result can be built by RELINKING existing nodes, with zero new node allocation.

### Core Mechanism
- Same core comparison loop as the array merge: at each step, compare the two lists' current front values, relink the smaller one onto the result, and advance only that list's pointer.
- **The key difference from the array version:** instead of writing VALUES into a freshly allocated result array, this simply repoints existing nodes' `next` pointers — the nodes themselves are reused, not copied.
- Once one list is exhausted, attach the other list's remainder directly (`tail->next = l1 ? l1 : l2`) — since the remaining nodes are ALREADY linked to each other in sorted order, this is a single pointer assignment, not a loop, unlike the array version's "drain the leftovers" loops.
- **Complexity: O(n+m) time, O(1) EXTRA space** (beyond the dummy node used for clean bookkeeping) — a genuinely better space profile than the array version, precisely because relinking existing structure costs nothing extra, while the array version had to allocate a whole new result array.

### Common Pitfalls
- Allocating brand new nodes with the merged values instead of relinking the EXISTING nodes from `l1`/`l2` — this works correctly but wastes the exact advantage linked lists have over arrays for this operation, turning an O(1)-extra-space merge into an unnecessary O(n+m)-extra-space one.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct ListNode {
    int val;
    ListNode* next;
    ListNode(int v) : val(v), next(nullptr) {}
};

ListNode* buildList(vector<int>& vals) {
    ListNode dummy(0);
    ListNode* tail = &dummy;
    for (int v : vals) { tail->next = new ListNode(v); tail = tail->next; }
    return dummy.next;
}

void printList(ListNode* head) {
    while (head) { cout << head->val; if (head->next) cout << " -> "; head = head->next; }
    cout << "\n";
}

// MERGE TWO SORTED LISTS: the exact same two-pointer merge logic from
// 02_Sliding_Window_and_Prefix_Sum's array-merge, adapted to linked lists -- with the
// dummy-head technique (Section 6) making the "build the result list" bookkeeping clean.
ListNode* mergeTwoLists(ListNode* l1, ListNode* l2) {
    ListNode dummy(0);
    ListNode* tail = &dummy;

    while (l1 != nullptr && l2 != nullptr) {
        if (l1->val <= l2->val) {
            tail->next = l1;   // RELINK existing nodes -- no new node allocation needed,
            l1 = l1->next;      // unlike the array version, which had to WRITE values into
        } else {                // a new array. Linked lists let you just rewire pointers.
            tail->next = l2;
            l2 = l2->next;
        }
        tail = tail->next;
    }

    // attach whichever list still has leftovers -- same "drain the remainder" idea as
    // the array merge, but here it's a single pointer reassignment, not a loop, since
    // the remaining nodes are ALREADY linked to each other
    tail->next = (l1 != nullptr) ? l1 : l2;

    return dummy.next;
}

int main() {
    vector<int> vals1{1, 2, 4};
    vector<int> vals2{1, 3, 4};
    ListNode* l1 = buildList(vals1);
    ListNode* l2 = buildList(vals2);

    ListNode* merged = mergeTwoLists(l1, l2);
    printList(merged);   // expected: 1 -> 1 -> 2 -> 3 -> 4 -> 4

    // O(n+m) time, O(1) EXTRA space (beyond the dummy node) -- genuinely better than the
    // array version's space profile, because linked lists let you relink existing nodes
    // instead of copying values into a freshly allocated result array

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Doubly Linked List — O(1) Removal With a Known Node

### The Why
A singly linked list's fundamental limitation: removing a node requires knowing its PREDECESSOR (to relink `prev->next` around the removed node), and a singly linked node has no way to find its own predecessor without a full O(n) traversal from the head. A doubly linked list fixes this directly, at the cost of one extra pointer per node.

### Core Mechanism
- Each node holds pointers to BOTH the next AND previous nodes.
- **O(1) removal given a pointer to the node:** since `node->prev` already IS the predecessor, removal is just two pointer rewires (`node->prev->next = node->next` and `node->next->prev = node->prev`), with explicit handling for the edge cases where the removed node is the current head or tail.
- This is exactly why a doubly linked list is the structure behind the classic LRU (Least Recently Used) cache: combined with a hash map for O(1) lookup of WHICH node corresponds to a given key, you get O(1) lookup AND O(1) removal-and-reinsertion of an arbitrary node — something neither structure could achieve alone (a hash map alone has no notion of "recency order," and a plain singly linked list alone can't remove an arbitrary node in O(1)).
- The cost: one extra pointer per node (roughly double the pointer overhead of a singly linked list) — a real but usually acceptable trade for the O(1) arbitrary-removal capability.

### Common Pitfalls
- Forgetting to update BOTH the `head` and `tail` pointers when removing the first or last node respectively — a doubly linked list has twice the pointer bookkeeping of a singly linked list, and it's easy to update the `next`/`prev` chain correctly while forgetting the class's own `head`/`tail` member variables need updating too when the removed node was one of those boundary nodes.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// DOUBLY LINKED LIST: each node holds pointers to BOTH the next AND previous nodes.
// The payoff: O(1) removal of a node GIVEN A POINTER TO IT, with no need to separately
// track or search for "the previous node" -- it's already right there.
struct DNode {
    int val;
    DNode *prev, *next;
    DNode(int v) : val(v), prev(nullptr), next(nullptr) {}
};

class DoublyLinkedList {
    DNode *head, *tail;
public:
    DoublyLinkedList() : head(nullptr), tail(nullptr) {}

    DNode* pushBack(int val) {
        DNode* node = new DNode(val);
        if (tail == nullptr) {
            head = tail = node;
        } else {
            tail->next = node;
            node->prev = tail;
            tail = node;
        }
        return node;   // return the new node so the caller can hold onto it for O(1)
                         // removal later -- this is the whole point of a doubly linked list
    }

    // O(1) removal GIVEN a pointer to the node -- no traversal needed to find its
    // predecessor, because node->prev already IS that predecessor
    void remove(DNode* node) {
        if (node->prev != nullptr) node->prev->next = node->next;
        else head = node->next;   // node was the head -- update head directly

        if (node->next != nullptr) node->next->prev = node->prev;
        else tail = node->prev;   // node was the tail -- update tail directly

        delete node;
    }

    void printForward() {
        DNode* cur = head;
        while (cur) { cout << cur->val; if (cur->next) cout << " <-> "; cur = cur->next; }
        cout << "\n";
    }

    void printBackward() {
        DNode* cur = tail;
        while (cur) { cout << cur->val; if (cur->prev) cout << " <-> "; cur = cur->prev; }
        cout << "\n";
    }
};

int main() {
    DoublyLinkedList list;
    list.pushBack(1);
    DNode* two = list.pushBack(2);
    list.pushBack(3);
    DNode* four = list.pushBack(4);
    list.pushBack(5);

    cout << "forward:  "; list.printForward();
    cout << "backward: "; list.printBackward();

    // O(1) removal -- no traversal needed, since we already hold direct pointers
    list.remove(two);
    list.remove(four);
    cout << "after removing 2 and 4:\n";
    cout << "forward:  "; list.printForward();

    // WHY THIS MATTERS: a SINGLY linked list can't do this -- removing a node requires
    // knowing its PREDECESSOR (to relink prev->next), and a singly linked node has no
    // way to find its own predecessor without a full traversal from the head, O(n).
    // A doubly linked list trades extra memory (one more pointer per node) for turning
    // that O(n) into O(1) -- this is exactly why LRU caches (a classic interview
    // structure) are built on a doubly linked list combined with a hash map: O(1) removal
    // of an arbitrary node, O(1) lookup of WHICH node to remove, combined together

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. When a Linked List Is (and Isn't) the Right Tool

### The Why
Linked lists are frequently taught in a vacuum, disconnected from when you'd actually reach for one over `vector` or `deque`. Closing that gap is more useful than knowing the operations in isolation.

### Core Mechanism
- **A linked list is the right tool when:** insertions/deletions happen frequently at positions you already hold a pointer to (not positions you need to search for), and random access by index is rarely or never needed. The LRU cache pattern (Section 7) is the clearest canonical example — frequent arbitrary-position removal, paired with a hash map to avoid ever needing to search.
- **A linked list is the WRONG tool when:** you need random access by index (`vector` wins, O(1) vs O(n)), or when the workload is dominated by appending to the end with occasional full scans (`vector`'s cache locality tends to win here too, despite matching or worse Big-O, for the same reasons discussed in `04_Sorting_Algorithms`'s heap sort cache-locality discussion — contiguous memory access is faster in practice than pointer-chasing, even at equal asymptotic complexity).
- **A fast self-check:** if the phrase "given a pointer to a specific node" would appear naturally in the problem's ideal solution, a linked list (possibly doubly linked, if removal from the middle matters) is worth considering. If the problem is about "the Kth element" or "sort this data," reach for `vector`, `deque`, or the sorting techniques from `04_Sorting_Algorithms` instead.
- In modern C++, `std::list` (`03_STL_Deep_Dive`, Section 6) already provides both singly-and-doubly-linked-adjacent functionality — implementing your own linked list by hand (as this notebook does) is for learning the mechanics; reach for `std::list` in real code, same guidance as every other from-scratch implementation in this course.

### Common Pitfalls
- Choosing a linked list by default for "a list of things" without considering the actual access pattern — modern hardware's cache behavior means `vector` often outperforms a linked list in practice even for operations where the linked list's Big-O looks equal or better, purely due to memory locality.


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Linked List Operation Cheat Sheet

| Operation | Singly Linked | Doubly Linked | Notes |
|---|---|---|---|
| Access by index | O(n) | O(n) | No random access in either |
| Insert at head | O(1) | O(1) | |
| Insert at tail | O(n), or O(1) with a maintained tail pointer | O(1) with a maintained tail pointer | |
| Delete by value | O(n) search + O(1) removal | O(n) search + O(1) removal | |
| Delete given a node pointer | O(n) — must find the predecessor | O(1) — predecessor is `node->prev` | The core doubly-linked advantage |
| Reverse | O(n) time; O(1) space iterative, O(n) space recursive | O(n) time, similar space trade-off | |
| Find middle / detect cycle | O(n) time, O(1) space (fast/slow) | Same | |

### Checkpoint Questions
1. State the core trade-off between `vector` and a linked list in one sentence.
2. Why is inserting at the head O(1) but inserting at the tail O(n), without a maintained tail pointer?
3. Concretely, why does recursive linked-list reversal cost O(n) extra space while iterative reversal costs O(1)?
4. What specific problem does the dummy head node solve, and what would the code look like without it (in general terms)?
5. Why can merging two sorted linked lists achieve O(1) extra space, while merging two sorted arrays cannot?
6. Why can a singly linked list not achieve O(1) removal given only a pointer to the node to remove, while a doubly linked list can?

### Answer Key
1. A linked list trades away O(1) random access (vector's strength) in exchange for O(1) insertion/deletion at an already-known position (vector's weakness, since vector must shift subsequent elements).
2. The head is, by definition, already directly accessible — no traversal needed to reach it. The tail requires walking the entire chain to find it first, UNLESS a separate tail pointer is maintained specifically to avoid that traversal.
3. Recursive reversal makes one recursive call per node before any pointer rewiring happens, and each of those calls has its own stack frame that stays alive until the base case is reached and unwinding begins — n nodes means n live stack frames, O(n) space. Iterative reversal uses a fixed three pointer variables (`prev`, `cur`, `next`) regardless of list length — O(1) space.
4. It eliminates the need for special-case code whenever an operation might need to modify or remove the ORIGINAL head node — without it, code would need a separate branch checking "is this the head?" before every removal/insertion, since modifying the head requires updating an external variable, while modifying elsewhere only requires updating a `next` pointer. The dummy node makes both cases identical by ensuring "the node before the real head" always exists as an ordinary node.
5. Linked lists let you RELINK existing nodes into the merged result (just rewiring `next` pointers) instead of copying values into a freshly allocated structure. Arrays have no equivalent — merging into a new array necessarily requires writing every value somewhere, which costs O(n+m) space for the result.
6. Removing a node requires updating its PREDECESSOR's `next` pointer to skip over it. A singly linked node has no way to find its own predecessor without a full traversal from the head (O(n)). A doubly linked node stores `prev` directly, making the predecessor immediately available — no search needed, hence O(1).

### Mini Project
Implement an LRU (Least Recently Used) cache with O(1) `get(key)` and O(1) `put(key, value)`.
1. Combine a doubly linked list (Section 7) — ordered by recency, most-recently-used at one end — with a hash map (`03_Hashing_and_Binary_Search`) mapping each key directly to its node in the list.
2. On `get`, use the hash map for O(1) lookup of the node, then move that node to the most-recently-used end of the list using Section 7's O(1) removal-and-reinsertion.
3. On `put`, if the cache is at capacity, remove the least-recently-used node (the other end of the list) — O(1) since it's a known position — and evict its corresponding entry from the hash map too.

This exercises: doubly linked list O(1) removal, combined with a hash map for O(1) lookup — the canonical example of this notebook's structure paired with `03_Hashing_and_Binary_Search`'s, each covering exactly what the other can't do alone.


## 10. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Basic Operations

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 707 | Design Linked List | Medium | Implement the full Section 1-2 operation set as a class — a good comprehensive fluency check |
| 203 | Remove Linked List Elements | Easy | Section 5's `removeElements`, directly |
| 83 | Remove Duplicates from Sorted List | Easy | Adjacent-duplicate removal, similar shape to `01_Arrays_and_Two_Pointers`' array version but with pointer relinking instead of a write-pointer |

### Reversal

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 206 | Reverse Linked List | Easy | Section 3, directly — implement both the iterative and recursive versions for practice |
| 92 | Reverse Linked List II | Medium | Reverse only a SUB-RANGE of the list — combine the dummy head technique (Section 5) with Section 3's three-pointer reversal, applied to just the target range |
| 25 | Reverse Nodes in k-Group | Hard | Repeated application of Section 3's reversal to successive groups of k nodes — carefully manage the connections between consecutive reversed groups |

### Fast/Slow Pointers

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 876 | Middle of the Linked List | Easy | Section 4, directly |
| 141 | Linked List Cycle | Easy | Section 4, directly — already referenced in `01_Arrays_and_Two_Pointers`' LeetCode set, now in its proper home |
| 142 | Linked List Cycle II | Medium | Find the cycle's ENTRANCE — the same two-phase Floyd's technique from `01_Arrays_and_Two_Pointers` Section 4, now applied to an actual linked list instead of an array |
| 234 | Palindrome Linked List | Easy | Find the middle (Section 4), reverse the second half (Section 3), then compare both halves — a clean combination of two techniques from this notebook |

### Dummy Head Node

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 19 | Remove Nth Node From End of List | Medium | Section 5's `removeNthFromEnd`, directly |
| 2 | Add Two Numbers | Medium | Build the result list digit-by-digit using a dummy head, handling carry propagation as you go |

### Merging and Combining Techniques

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 21 | Merge Two Sorted Lists | Easy | Section 6, directly |
| 23 | Merge k Sorted Lists | Hard | Already in `01_Heaps_and_Priority_Queues`' LeetCode set (Section 6, k-way merge) — the heap-based approach generalizes Section 6's two-list merge to k lists |
| 146 | LRU Cache | Medium | This notebook's Mini Project — doubly linked list + hash map, directly |

### Self-Check Before Moving to `08_Stacks_Queues_Deques`
If you can implement iterative reversal and the dummy-head removal pattern without hesitation, and you can explain why a doubly linked list enables O(1) arbitrary removal when a singly linked list can't, you've internalized this notebook's two most-tested ideas. `08_Stacks_Queues_Deques` closes out Phase 02 by formalizing structures you've already been using informally (the call stack throughout `02_Recursion`, the queue inside every BFS you've seen referenced) — a good moment to notice how much of this course has been quietly building toward ideas it hasn't explicitly named yet.
